# FoldX Mutation Panel

Ноутбук для чистого FoldX-скрининга одной белковой цепи в белок-белковом комплексе.

Что делает этот workflow:
- принимает `PDB ID` комплекса;
- выбирает одну мутируемую белковую цепь;
- строит для каждой выбранной позиции все 19 не-WT замен через FoldX;
- сохраняет прогресс по кейсам и умеет продолжать при повторном запуске;
- считает две FoldX-метрики: `binding_ddg` и `stability_ddg`;
- строит итоговый ranking по выбранной пользователем метрике;
- даёт визуальное сравнение `WT complex` и `FoldX mutant`.

Важно:
- `binding_ddg` имеет смысл в первую очередь для интерфейсных позиций;
- `stability_ddg` подходит для полного сканирования всей цепи.

OpenFold3 здесь не используется.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent
HELPERS_DIR = NOTEBOOK_ROOT / 'helpers'
OPENFOLD_REPO_DIR = REPO_ROOT / 'openfold-3'

for path in (NOTEBOOK_ROOT, HELPERS_DIR, OPENFOLD_REPO_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from of_notebook_lib import preview_molecules
from of_notebook_lib.ddg_panel import (
    DEFAULT_SAFE_PPI_TARGET,
    build_foldx_run_name,
    load_foldx_panel_visual_rows,
    preview_foldx_panel_input,
    render_foldx_structure_comparison_html,
    render_info_card,
)
from openfold3.benchmark.foldx_panel import run_foldx_panel

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 200)


## 1. Проверка FoldX

Здесь проверяется только наличие бинарника FoldX. Никакие OpenFold3-модели не поднимаются.

In [ ]:
available_cpus = os.cpu_count() or 1
foldx_candidates = [
    os.environ.get('FOLDX_BINARY'),
    str(REPO_ROOT / 'tools' / 'bin' / 'foldx'),
]
foldx_resolved = next((candidate for candidate in foldx_candidates if candidate and Path(candidate).exists()), None)
display(HTML(render_info_card(
    'FoldX runtime',
    [
        ('FOLDX_BINARY', os.environ.get('FOLDX_BINARY')),
        ('resolved_binary', foldx_resolved),
        ('available_cpus', available_cpus),
        ('cpu_command', 'nproc'),
    ],
    accent='#1c7ed6' if foldx_resolved else '#c92a2a',
)))


## 2. Основные параметры

Пользователь меняет только `PDB ID`, цепь, позиции и метрику ранжирования.

In [ ]:
pdb_id = DEFAULT_SAFE_PPI_TARGET['pdb_id']
mutable_chain_id = DEFAULT_SAFE_PPI_TARGET['mutable_chain_id']

# positions_mode: 'explicit_list' or 'all_chain_positions'
positions_mode = 'explicit_list'
positions_text = '35-45'

# ranking_metric: 'stability_ddg' or 'binding_ddg'
# Use stability_ddg for full-chain scans. Use binding_ddg for interface-focused scans.
ranking_metric = 'stability_ddg'

# Recommended starting point: 2-4 workers. Keep 1 for easiest debugging.
num_workers = 1

experiment_name = build_foldx_run_name(pdb_id=pdb_id, mutable_chain_id=mutable_chain_id)
output_parent = REPO_ROOT / 'results' / 'foldx_panels'
pdb_cache_dir = NOTEBOOK_ROOT / '.pdb_cache'

## 3. Предпросмотр входа

Notebook загружает `PDB ID`, показывает состав комплекса и считает объём панели `19xN`.

Важно:
- `position_1based` ниже это индекс в разрешённой FoldX-последовательности цепи;
- `residue_id` это реальный номер остатка в структуре;
- `is_interface_8a=True` означает, что позиция находится близко к partner chain и для неё осмыслен `binding_ddg`.

In [ ]:
resolved_molecules, pdb_preview_df, panel_preview_df, panel_summary_input, positions = preview_foldx_panel_input(
    pdb_id=pdb_id,
    mutable_chain_id=mutable_chain_id,
    positions_mode=positions_mode,
    positions_text=positions_text,
    pdb_cache_dir=pdb_cache_dir,
)

display(HTML(render_info_card(
    'FoldX panel plan',
    [
        ('pdb_id', panel_summary_input['pdb_id']),
        ('mutable_chain_id', panel_summary_input['mutable_chain_id']),
        ('resolved_sequence_length', panel_summary_input['sequence_length']),
        ('first_residue_id', panel_summary_input['first_residue_id']),
        ('last_residue_id', panel_summary_input['last_residue_id']),
        ('interface_positions', panel_summary_input['interface_positions']),
        ('ranking_metric', ranking_metric),
        ('positions_count', panel_summary_input['positions_count']),
        ('planned_mutants', panel_summary_input['planned_mutants']),
    ],
    accent='#2f9e44',
)))

display(pdb_preview_df)
display(preview_molecules(resolved_molecules))
display(panel_preview_df.head(50))
print(f"Resolved positions span: {positions[0]}..{positions[-1]}")

## 4. Подготовка run-root

Если перезапустить эту ячейку, будет создан новый `run_root`. Чтобы продолжить старый запуск, ниже можно явно указать существующий путь.

In [ ]:
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_root = output_parent / f'{experiment_name}_{run_stamp}'
run_root.mkdir(parents=True, exist_ok=True)

plan_payload = {
    'run_root': str(run_root),
    'pdb_id': pdb_id,
    'mutable_chain_id': mutable_chain_id,
    'positions': list(positions),
    'planned_mutants': panel_summary_input['planned_mutants'],
}
(run_root / 'plan.json').write_text(json.dumps(plan_payload, indent=2), encoding='utf-8')

display(HTML(render_info_card(
    'Run root prepared',
    [
        ('run_root', run_root),
        ('plan_json', run_root / 'plan.json'),
        ('planned_mutants', panel_summary_input['planned_mutants']),
    ],
    accent='#7048e8',
)))


## 5. Запуск FoldX-панели

Если `run_root` уже содержит рассчитанные кейсы, `run_foldx_panel()` использует их повторно и досчитает только отсутствующие.

Для проверки числа CPU в терминале можно выполнить `nproc`. В ноутбуке это число уже показано в блоке `FoldX runtime`.

Если вы меняете `ranking_metric`, лучше использовать новый `run_root`, чтобы summary и ranking были пересчитаны согласованно.

In [ ]:
# To resume an earlier run, replace run_root with an existing path before executing this cell.
# run_root = Path('/abs/path/to/existing_foldx_run_root')

foldx_result = run_foldx_panel(
    output_root=run_root,
    pdb_id=pdb_id,
    chain_id=mutable_chain_id,
    positions=positions,
    cache_dir=pdb_cache_dir,
    num_workers=num_workers,
    ranking_metric=ranking_metric,
)

summary_payload = json.loads(foldx_result.summary_json_path.read_text(encoding='utf-8'))
ranking_df = pd.read_csv(foldx_result.ranking_csv_path) if foldx_result.ranking_csv_path.exists() and foldx_result.ranking_csv_path.stat().st_size else pd.DataFrame()
rows_df = pd.read_csv(foldx_result.rows_csv_path) if foldx_result.rows_csv_path.exists() and foldx_result.rows_csv_path.stat().st_size else pd.DataFrame()

display(HTML(render_info_card(
    'FoldX run finished',
    [
        ('run_root', foldx_result.output_root),
        ('summary_json', foldx_result.summary_json_path),
        ('rows_csv', foldx_result.rows_csv_path),
        ('ranking_csv', foldx_result.ranking_csv_path),
        ('ranking_metric', summary_payload['ranking_metric']),
        ('num_workers', summary_payload['num_workers']),
        ('total_mutations', summary_payload['total_mutations']),
        ('successful_mutations', summary_payload['successful_mutations']),
    ],
    accent='#2f9e44',
)))

display(ranking_df.head(50))

## 6. Итоговые таблицы

Первая таблица показывает глобальный ranking по выбранной `ranking_metric`.

Ключевые колонки:
- `foldx_stability_ddg_kcal_mol`: изменение энергии после мутации в модели FoldX;
- `foldx_binding_ddg_kcal_mol`: изменение интерфейсной энергии комплекса;
- `foldx_score_kcal_mol`: та же метрика, которая использована для ranking.

In [ ]:
display(Markdown('### Global ranking'))
display(ranking_df.head(100))

display(Markdown('### All mutation rows'))
display(rows_df.head(100))


## 7. Визуальное сравнение WT и FoldX mutant

Эта секция позволяет выбрать конкретную мутацию и сравнить WT complex с локально собранным FoldX mutant.

В заголовке viewer выводятся сразу три числа: `selected`, `binding`, `stability`.

In [ ]:
visual_rows = load_foldx_panel_visual_rows(foldx_result.summary_json_path)
print(f'visual rows: {len(visual_rows)}')

try:
    import ipywidgets as widgets

    options = [
        (f"{row.chain_id}:{row.from_residue}{row.position_1based}{row.to_residue}", row.case_id)
        for row in visual_rows
        if row.foldx_mutant_model_path is not None
    ]
    picker = widgets.Dropdown(options=options, description='Mutant')
    output = widgets.Output()

    def refresh(*_args):
        selected = next(row for row in visual_rows if row.case_id == picker.value)
        with output:
            output.clear_output()
            display(HTML(render_foldx_structure_comparison_html(selected)))

    picker.observe(refresh, names='value')
    refresh()
    display(widgets.VBox([picker, output]))
except Exception:
    if visual_rows:
        display(HTML(render_foldx_structure_comparison_html(visual_rows[0])))
    else:
        display(Markdown('Нет доступных FoldX-структур для viewer.'))
